# <font color="#418FDE" size="6.5" uppercase>**Robust Sparse**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply sparse linear estimators for feature selection and compact prediction. 
- Use Bayesian linear models to estimate regularized coefficients and uncertainty-related behavior. 
- Fit robust regressors that reduce sensitivity to outliers. 


## **1. Sparse Linear Methods**

### **1.1. Least Angle Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_01.jpg?v=1783960018" width="250">



>* Builds sparse models by adding key predictors
>* Balances tied features for compact prediction

>* Builds models from sparse to complex
>* Highlights key features in high dimensions

>* Tracks coefficient changes and predictor competition
>* Supports balanced sparsity with correlated features



In [ ]:
#@title Python Code - Least Angle Regression

# Demonstrate Least Angle Regression manually.
# Sparse paths reveal feature entry order.
# Tiny synthetic data keeps output readable.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducibility.
rng = np.random.default_rng(7)

# Create compact correlated predictor data.
n_samples, n_features = 40, 6
X_raw = rng.normal(size=(n_samples, n_features))

# Add realistic feature correlation structure.
X_raw[:, 1] = 0.75 * X_raw[:, 0] + rng.normal(size=n_samples) * 0.35
X_raw[:, 3] = 0.60 * X_raw[:, 2] + rng.normal(size=n_samples) * 0.45

# Define only two truly useful coefficients.
true_beta = np.array([2.4, 0.0, -1.7, 0.0, 0.0, 0.9])
y_raw = X_raw @ true_beta + rng.normal(size=n_samples) * 0.45

# Standardize predictors and response safely.
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0, ddof=0)
X = (X_raw - X_mean) / np.maximum(X_std, 1e-12)

# Center response for intercept-free path demonstration.
y = y_raw - y_raw.mean()
assert X.shape == (n_samples, n_features)

# Prepare a simple Least Angle Regression path.
beta = np.zeros(n_features)
active = []
path = [beta.copy()]

# Run a few sparse path steps.
for step in range(4):
    residual = y - X @ beta
    correlations = X.T @ residual
    inactive = [j for j in range(n_features) if j not in active]

    # Add the strongest inactive predictor.
    chosen = max(inactive, key=lambda j: abs(correlations[j]))
    active.append(chosen)
    Xa = X[:, active]

    # Move toward least-squares fit on active features.
    beta_active = np.linalg.lstsq(Xa, y, rcond=None)[0]
    beta[active] = beta_active
    path.append(beta.copy())

# Convert path for compact plotting.
path = np.array(path)
selected_names = [f"x{j}" for j in active]

# Print concise teaching summary.
print("Manual LAR-style sparse path demonstration")
print(f"Feature entry order: {selected_names}")
print(f"Nonzero final coefficients: {np.count_nonzero(np.round(beta, 6))}")
print(f"True useful features: ['x0', 'x2', 'x5']")

# Plot coefficient growth along the sparse path.
plt.figure(figsize=(7, 4))
for j in range(n_features):
    plt.plot(range(len(path)), path[:, j], marker="o", label=f"x{j}")

# Label the single teaching plot clearly.
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Sparse path step")
plt.ylabel("Standardized coefficient")
plt.title("Least Angle Regression idea: coefficients enter gradually")
plt.legend(ncol=3, fontsize=8)

# Display the compact coefficient path plot.
plt.tight_layout()
plt.show()



### **1.2. Lasso LARS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_02.jpg?v=1783960014" width="250">



>* Builds compact models using selected predictors
>* Sets irrelevant feature coefficients to zero

>* Traces models across changing regularization levels
>* Adds or removes predictors along the path

>* Efficiently traces sparse model solution paths
>* Interpret selected features with validation and caution



### **1.3. Orthogonal Matching Pursuit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_03.jpg?v=1783960016" width="250">



>* Selects useful features one at a time
>* Refits coefficients for compact prediction

>* Residuals become orthogonal to selected features
>* Correlated predictors can affect feature choices

>* Choose stopping rules to balance fit
>* Validate greedy selections for reliable sparse models



In [ ]:
#@title Python Code - Orthogonal Matching Pursuit

# Demonstrate sparse feature selection.
# Build Orthogonal Matching Pursuit manually.
# Keep output short and visual.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible synthetic regression data.
rng = np.random.default_rng(7)
samples, features = 80, 12
X = rng.normal(size=(samples, features))

# Standardize features for fair matching.
X = (X - X.mean(axis=0)) / X.std(axis=0)
true_coef = np.zeros(features)
true_coef[[1, 4, 9]] = [3.0, -2.0, 1.5]

# Add small noise to targets.
y = X @ true_coef + rng.normal(scale=0.4, size=samples)
assert X.shape == (samples, features)

# Start with no selected features.
selected = []
residual = y.copy()
coef = np.zeros(features)

# Greedily select three useful features.
for step in range(3):
    scores = np.abs(X.T @ residual)
    scores[selected] = -np.inf
    chosen = int(np.argmax(scores))

    selected.append(chosen)
    X_selected = X[:, selected]
    beta, *_ = np.linalg.lstsq(X_selected, y, rcond=None)
    residual = y - X_selected @ beta

# Store final sparse coefficients.
coef[selected] = beta
predictions = X @ coef
rmse = np.sqrt(np.mean((y - predictions) ** 2))

# Print compact teaching summary.
print(f"Selected feature indexes: {selected}")
print(f"True nonzero indexes: {[1, 4, 9]}")
print(f"Nonzero coefficient count: {np.count_nonzero(coef)}")
print(f"Training RMSE: {rmse:.3f}")

# Plot true and learned coefficients.
plt.figure(figsize=(8, 4))
plt.stem(true_coef, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="True")
plt.stem(coef, linefmt="C1--", markerfmt="C1s", basefmt=" ", label="OMP")
plt.xlabel("Feature index")
plt.ylabel("Coefficient value")
plt.title("Orthogonal Matching Pursuit selects a compact model")
plt.legend()
plt.tight_layout()
plt.show()



## **2. Bayesian Linear Models**

### **2.1. Bayesian Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_01.jpg?v=1783960036" width="250">



>* Probabilistic ridge treats coefficients as uncertain
>* Shrinkage reduces noise while preserving patterns

>* Learns regularization strength from data
>* Shrinks weak predictors for reliable predictions

>* Shows confidence levels for coefficient estimates
>* Supports cautious predictions with imperfect data



In [ ]:
#@title Python Code - Bayesian Ridge

# Demonstrate Bayesian Ridge without external machine learning libraries.
# Synthetic housing data keeps the example small.
# Coefficient uncertainty is approximated from posterior covariance.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable teaching results.
rng = np.random.default_rng(42)

# Create correlated house features in simple units.
n_samples, n_features = 80, 3
size_sqft = rng.normal(1800, 350, n_samples)
rooms = size_sqft / 450 + rng.normal(0, 0.6, n_samples)
age_years = rng.normal(25, 10, n_samples)

# Stack features and create noisy prices.
X = np.column_stack([size_sqft, rooms, age_years])
true_coef = np.array([0.12, 18.0, -1.5])
y = X @ true_coef + rng.normal(0, 25, n_samples)

# Validate the tiny design matrix shape.
assert X.shape == (n_samples, n_features)
assert y.shape == (n_samples,)

# Standardize features for stable Bayesian calculations.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

# Center the target like an intercept model.
y_mean = y.mean()
y_centered = y - y_mean

# Choose simple Bayesian Ridge precision values.
alpha_noise = 1.0 / np.var(y_centered)
lambda_prior = 1.0

# Compute posterior covariance and mean analytically.
precision = lambda_prior * np.eye(n_features)
posterior_cov = np.linalg.inv(precision + alpha_noise * X_scaled.T @ X_scaled)
posterior_mean = alpha_noise * posterior_cov @ X_scaled.T @ y_centered

# Convert posterior spread into coefficient uncertainty.
coef_std = np.sqrt(np.diag(posterior_cov))
feature_names = ["size_sqft", "rooms", "age_years"]

# Predict using the posterior mean coefficients.
y_pred = y_mean + X_scaled @ posterior_mean
rmse = np.sqrt(np.mean((y - y_pred) ** 2))

# Print compact results for beginner interpretation.
print("Bayesian Ridge from NumPy, no sklearn used.")
print(f"RMSE on tiny training data: {rmse:.2f}")
print("Feature coefficients show shrinkage and uncertainty:")

# Print one short line per coefficient.
for name, mean, spread in zip(feature_names, posterior_mean, coef_std):
    print(f"{name}: mean={mean:.2f}, uncertainty={spread:.2f}")

# Plot coefficient means with uncertainty bars.
plt.figure(figsize=(7, 4))
plt.errorbar(feature_names, posterior_mean, yerr=coef_std, fmt="o", capsize=6)
plt.axhline(0, color="gray", linewidth=1)
plt.title("Bayesian Ridge coefficient uncertainty")
plt.ylabel("Standardized coefficient value")
plt.tight_layout()
plt.show()



### **2.2. ARD Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_02.jpg?v=1783960028" width="250">



>* Bayesian model learns feature relevance automatically
>* Separate shrinkage switches weak coefficients off

>* Learns separate shrinkage for each feature
>* Keeps useful predictors, suppresses noisy ones

>* Estimates coefficient uncertainty, not just values
>* Shrunk coefficients need careful interpretation



In [ ]:
#@title Python Code - ARD Regression

# Demonstrate ARD regression with adaptive shrinkage.
# Use only NumPy for transparent learning.
# Compare relevant and irrelevant feature weights.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic synthetic housing style data.
rng = np.random.default_rng(7)
n_samples, n_features = 80, 8

# Generate standardized candidate predictors.
X = rng.normal(size=(n_samples, n_features))
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Define only three truly useful coefficients.
true_w = np.array([3.0, 0.0, -2.0, 0.0, 1.5, 0.0, 0.0, 0.0])
y = X @ true_w + rng.normal(scale=0.8, size=n_samples)

# Validate the small teaching dataset shape.
assert X.shape == (n_samples, n_features)
assert y.shape == (n_samples,)

# Initialize Bayesian precision values for ARD.
alpha = np.ones(n_features)
beta = 1.0 / np.var(y)

# Run a compact fixed point ARD update.
for step in range(80):
    precision = np.diag(alpha) + beta * X.T @ X
    covariance = np.linalg.inv(precision)
    mean_w = beta * covariance @ X.T @ y
    gamma = 1.0 - alpha * np.diag(covariance)
    alpha = gamma / (mean_w**2 + 1e-8)
    residual = y - X @ mean_w
    beta = (n_samples - gamma.sum()) / (residual @ residual + 1e-8)

# Convert precision into uncertainty style summaries.
std_w = np.sqrt(np.diag(covariance))
active = np.abs(mean_w) > 0.25

# Print concise interpretation for learners.
print("ARD keeps coefficients with evidence, shrinks weak ones.")
print(f"Estimated active features: {np.where(active)[0].tolist()}")
print(f"True active features: {[0, 2, 4]}")
print(f"Noise precision estimate: {beta:.2f}")

# Plot coefficient estimates with uncertainty bars.
plt.figure(figsize=(8, 4))
plt.errorbar(range(n_features), mean_w, yerr=std_w, fmt="o", capsize=4)
plt.axhline(0, color="black", linewidth=1)
plt.xticks(range(n_features))
plt.xlabel("Feature index")
plt.ylabel("Estimated coefficient")
plt.title("ARD Regression: adaptive shrinkage and uncertainty")
plt.tight_layout()
plt.show()



### **2.3. Uncertainty Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_03.jpg?v=1783960033" width="250">



>* Coefficients have ranges, not fixed answers
>* Uncertainty shows evidence behind learned relationships

>* Coefficient uncertainty affects prediction confidence
>* Limited evidence should increase model caution

>* Bayesian models shrink uncertain coefficients cautiously
>* Uncertainty supports honest real-world decisions



## **3. Robust Regression**

### **3.1. Huber Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_01.jpg?v=1783960024" width="250">



>* Robust linear fitting for mostly reliable data
>* Limits outlier influence on model estimates

>* Small errors fit like standard regression
>* Large errors stay but influence less

>* Stable linear modeling with outlier resistance
>* Threshold choice controls residual downweighting



In [ ]:
#@title Python Code - Huber Regression

# Demonstrate Huber regression without external machine learning libraries.
# Compare ordinary least squares against robust fitting.
# Outliers should influence the robust line less.

# No extra installations are required for this example.

# Import small scientific and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Create deterministic data with several extreme outliers.
rng = np.random.default_rng(7)
x = np.linspace(0.0, 10.0, 45)
y = 3.0 + 2.0 * x + rng.normal(0.0, 1.0, x.size)

# Add unusual observations that distort ordinary least squares.
y[[6, 18, 34]] += np.array([18.0, -22.0, 20.0])
X = np.column_stack([np.ones_like(x), x])
assert X.shape[0] == y.size

# Fit ordinary least squares using NumPy linear algebra.
ols_coef = np.linalg.lstsq(X, y, rcond=None)[0]

# Define a compact Huber iteratively reweighted least squares solver.
def fit_huber(X, y, delta=2.0, steps=25):
    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    for step in range(steps):
        residuals = y - X @ coef
        scale = np.maximum(np.abs(residuals), 1e-8)
        weights = np.minimum(1.0, delta / scale)
        WX = X * weights[:, None]
        wy = y * weights
        coef = np.linalg.lstsq(WX, wy, rcond=None)[0]
    return coef

# Fit Huber regression with a moderate residual threshold.
huber_coef = fit_huber(X, y, delta=2.0, steps=25)

# Compare fitted slopes and intercepts in a few lines.
print("NumPy version:", np.__version__)
print("OLS intercept and slope:", np.round(ols_coef, 3))
print("Huber intercept and slope:", np.round(huber_coef, 3))
print("Huber keeps outliers, but reduces their leverage.")

# Draw the data and both fitted regression lines.
grid = np.linspace(x.min(), x.max(), 100)
G = np.column_stack([np.ones_like(grid), grid])
plt.figure(figsize=(7, 4))
plt.scatter(x, y, label="data with outliers", color="gray")

# Plot ordinary and robust fitted lines together.
plt.plot(grid, G @ ols_coef, label="ordinary least squares")
plt.plot(grid, G @ huber_coef, label="Huber regression")
plt.xlabel("machine age in years")
plt.ylabel("repair cost in dollars")
plt.title("Huber regression reduces outlier influence")
plt.legend()
plt.tight_layout()
plt.show()



### **3.2. RANSAC Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_02.jpg?v=1783960022" width="250">



>* Fits models from random data subsets
>* Finds majority pattern despite extreme outliers

>* Fit random clean subsets, test all data
>* Keep strongest consensus, refit using inliers

>* Tune thresholds and trials carefully
>* Best for one clear majority trend



In [ ]:
#@title Python Code - RANSAC Regression

# Demonstrate RANSAC regression with outliers.
# Compare ordinary least squares visually.
# Use only NumPy and Matplotlib.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible distance and time data.
rng = np.random.default_rng(7)

# Simulate typical trips in kilometers.
x = np.linspace(1, 20, 45)
y = 4.0 + 2.2 * x + rng.normal(0, 2.0, x.size)

# Add extreme delay outliers.
outlier_ids = np.array([5, 12, 28, 38])
y[outlier_ids] += np.array([35, 28, 42, 30])

# Validate the simple one-feature dataset.
assert x.shape == y.shape and x.size >= 20

# Fit ordinary least squares using all points.
ols_slope, ols_intercept = np.polyfit(x, y, 1)

# Configure a small RANSAC search.
best_count = -1
best_slope = 0.0
best_intercept = 0.0
threshold = 5.0

# Try many tiny random two-point models.
for trial in range(120):
    pair = rng.choice(x.size, size=2, replace=False)
    slope, intercept = np.polyfit(x[pair], y[pair], 1)
    residuals = np.abs(y - (slope * x + intercept))
    inliers = residuals < threshold

    # Keep the model with strongest consensus.
    if int(inliers.sum()) > best_count:
        best_count = int(inliers.sum())
        best_slope = float(slope)
        best_intercept = float(intercept)

# Refit final line using detected inliers.
final_residuals = np.abs(y - (best_slope * x + best_intercept))
final_inliers = final_residuals < threshold
ransac_slope, ransac_intercept = np.polyfit(x[final_inliers], y[final_inliers], 1)

# Print a compact comparison summary.
print(f"OLS slope: {ols_slope:.2f} minutes per km")
print(f"RANSAC slope: {ransac_slope:.2f} minutes per km")
print(f"RANSAC inliers: {final_inliers.sum()} of {x.size}")

# Draw data, outliers, and fitted lines.
plt.figure(figsize=(7, 4))
plt.scatter(x[final_inliers], y[final_inliers], label="RANSAC inliers")
plt.scatter(x[~final_inliers], y[~final_inliers], label="RANSAC outliers")
plt.plot(x, ols_slope * x + ols_intercept, label="OLS line")
plt.plot(x, ransac_slope * x + ransac_intercept, label="RANSAC line")

# Label the teaching plot clearly.
plt.xlabel("Trip distance, kilometers")
plt.ylabel("Travel time, minutes")
plt.title("RANSAC reduces outlier influence")
plt.legend()
plt.show()



### **3.3. Quantile Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_03.jpg?v=1783960020" width="250">



>* Models chosen outcome quantiles, not just averages
>* Reduces outlier influence in skewed data

>* Shows effects across outcome levels
>* Reveals data heterogeneity beyond averages

>* Handles meaningful outliers without model domination
>* Targets medians or upper ranges for decisions



In [ ]:
#@title Python Code - Quantile Regression

# Quantile regression estimates conditional distribution points.
# Median fitting reduces outlier influence strongly.
# This example uses only NumPy optimization.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Create deterministic synthetic housing-like data.
rng = np.random.default_rng(42)
x = np.linspace(500, 3000, 80)
noise = rng.normal(0, 25000, x.size)

# Build prices with several high outliers.
y = 60000 + 140 * x + noise
outlier_idx = np.array([8, 18, 65, 72])
y[outlier_idx] += np.array([250000, 180000, 320000, 260000])

# Validate matching one-dimensional sample sizes.
assert x.ndim == y.ndim == 1
assert x.size == y.size and x.size > 10

# Standardize feature for stable gradient steps.
x_mean = x.mean()
x_std = x.std()
z = (x - x_mean) / x_std

# Add intercept column for linear fitting.
X = np.column_stack([np.ones_like(z), z])
assert X.shape == (x.size, 2)

# Define quantile gradient descent solver.
def fit_quantile(X, y, q, lr=0.02, steps=6000):
    beta = np.zeros(X.shape[1])
    for step in range(steps):
        residual = y - X @ beta
        weights = q - (residual < 0).astype(float)
        grad = -(X.T @ weights) / y.size
        beta -= lr * grad
    return beta

# Fit mean regression and median quantile regression.
mean_beta = np.linalg.lstsq(X, y, rcond=None)[0]
median_beta = fit_quantile(X, y, q=0.50)
upper_beta = fit_quantile(X, y, q=0.90)

# Convert predictions back to dollar scale.
y_mean = X @ mean_beta
y_median = X @ median_beta
y_upper = X @ upper_beta

# Compare typical prediction near two thousand square feet.
home_sqft = 2000
home_z = (home_sqft - x_mean) / x_std
home_row = np.array([1.0, home_z])

# Print concise teaching summary lines.
print(f"NumPy version: {np.__version__}")
print(f"Mean model at {home_sqft} sq ft: ${home_row @ mean_beta:,.0f}")
print(f"Median quantile at {home_sqft} sq ft: ${home_row @ median_beta:,.0f}")
print(f"Upper quantile at {home_sqft} sq ft: ${home_row @ upper_beta:,.0f}")
print("Median quantile is less pulled by high-price outliers.")

# Plot data and three fitted regression lines.
plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=28, alpha=0.65, label="homes")
plt.plot(x, y_mean, color="red", label="mean regression")
plt.plot(x, y_median, color="green", label="median quantile")
plt.plot(x, y_upper, color="purple", label="90th quantile")

# Label the single teaching plot clearly.
plt.xlabel("Home size in square feet")
plt.ylabel("Sale price in dollars")
plt.title("Quantile Regression Reduces Outlier Dominance")
plt.legend()

# Display the completed visualization.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Robust Sparse**</font>


In this lecture, you learned to:
- Apply sparse linear estimators for feature selection and compact prediction. 
- Use Bayesian linear models to estimate regularized coefficients and uncertainty-related behavior. 
- Fit robust regressors that reduce sensitivity to outliers. 

In the next Lecture (Lecture B), we will go over 'Generalized Bases'